# Klasifikasi Kategori Produk untuk Pencatatan Keuangan UMKM
## PaddleOCR, Gemini, dan LSTM pada Nama Produk Hasil Pemindaian Struk

Alur utama notebook: **foto struk ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ OpenCV ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ PaddleOCR ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ Gemini (penyusunan struktur) ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ BiLSTM (kategori produk)**. Gemini tidak menentukan kategori; label kategori selalu diprediksi oleh BiLSTM.


---
## Konfigurasi penelitian
Notebook ini dirancang untuk **Google Colab**. Parameter dikumpulkan di satu sel agar mudah didokumentasikan dan diubah.

In [ ]:
CONFIG = {
    'dataset_filename': 'dataset_300.csv',
    'test_size': 0.20,
    'random_state': 42,
    'epochs': 20,
    'batch_size': 16,
    'max_words': 2000,
    'max_length': 12,
    'embedding_dim': 32,
    'lstm_units': 32,
    'minimum_confidence': 0.50,
}

print('Runtime: Google Colab')
print('Parameter:', CONFIG)

In [ ]:
# Jalankan sekali di Google Colab.
!pip -q install paddlepaddle==3.2.2 paddleocr==3.4.1 opencv-python scikit-learn tensorflow pandas matplotlib google-genai


> **Wajib:** setelah sel instalasi selesai, pilih **Runtime ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â ÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢ Restart session** di Google Colab. Jangan menjalankan ulang sel instalasi setelah `PaddleOCR` sudah diimpor, karena PaddleX tidak mendukung inisialisasi ulang pada runtime yang sama. Setelah restart, jalankan seluruh sel dari atas ke bawah sekali.

### Unggah dataset
Unggah file `dataset_300.csv` dari folder `backend_python/data` pada proyek Anda. Colab akan menyimpannya sementara di `/content`; unggah ulang jika runtime Colab direset.

In [ ]:
from pathlib import Path
from google.colab import files

uploaded = files.upload()
DATASET_PATH = Path('/content') / CONFIG['dataset_filename']
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'Harap unggah file bernama {CONFIG["dataset_filename"]}. File yang terbaca: {list(uploaded)}'
    )
print('Dataset siap:', DATASET_PATH)

---
# Tahap 1 ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â Business Understanding

Pencatatan transaksi UMKM dari struk belanja sering dilakukan manual. Tahap pertama adalah memperoleh teks dari foto struk dengan OCR; nama barang hasil OCR dapat bervariasi, singkat, atau mengandung salah ketik. Hal tersebut membuat pengelompokan transaksi ke kategori seperti `makanan`, `minuman`, `bahan_pokok`, dan `kebutuhan_rumah_tangga` memakan waktu.

**Tujuan penelitian:** menerapkan PaddleOCR untuk mengekstraksi teks pada struk dan membangun model yang mengklasifikasikan nama produk hasil OCR secara otomatis agar transaksi dapat dicatat dan dianalisis lebih cepat.

**Kriteria keberhasilan:** model dievaluasi dengan accuracy, precision, recall, F1-score, confusion matrix, dan waktu prediksi. Prediksi dengan keyakinan di bawah 50% akan diberi kategori `lainnya` pada aplikasi untuk ditinjau pengguna.

---
# Tahap 2 ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â Data Understanding
Dataset memiliki dua kolom: `text` (nama produk, termasuk variasi yang mungkin muncul dari OCR) dan `category` (label kategori). Sel berikut menampilkan ukuran data, sampel data, data kosong, dan persebaran kategori.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(DATASET_PATH)
print(f'Jumlah data: {len(df)}')
print('Kolom:', df.columns.tolist())
display(df.head(10))
print('\nNilai kosong:')
display(df.isna().sum().to_frame('jumlah'))

category_counts = df['category'].value_counts().sort_values(ascending=False)
display(category_counts.to_frame('jumlah_data'))
ax = category_counts.plot(kind='bar', color='#2563eb', figsize=(8, 4))
ax.set_title('Distribusi Data per Kategori')
ax.set_xlabel('Kategori')
ax.set_ylabel('Jumlah data')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

### Penyusunan struktur struk dengan Gemini

Notebook ini tidak menggunakan object detection/YOLO. Seluruh foto struk dibaca PaddleOCR; Gemini menyusun teks OCR menjadi merchant, tanggal, item, qty, harga, jumlah, dan total.


### Ekstraksi teks struk dengan PaddleOCR
Unggah satu foto struk saat sel berikut dijalankan. Sel memakai OpenCV, PaddleOCR berbahasa Inggris, dan ambang confidence 0,35. Sel ini opsional untuk pelatihan dataset, tetapi diperlukan untuk mendemonstrasikan alur input foto.

In [ ]:
import cv2
import numpy as np
from paddleocr import PaddleOCR

# Ubah menjadi True ketika ingin menguji foto struk.
UPLOAD_RECEIPT = False

if UPLOAD_RECEIPT:
    receipt_upload = files.upload()
    receipt_name = next(iter(receipt_upload))
    image = cv2.imdecode(np.frombuffer(receipt_upload[receipt_name], np.uint8), cv2.IMREAD_COLOR)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # PaddleOCR memerlukan array tiga kanal, jadi hasil grayscale dikonversi kembali ke BGR.
    prepared_gray = cv2.convertScaleAbs(gray, alpha=1.25, beta=5)
    prepared_image = cv2.cvtColor(prepared_gray, cv2.COLOR_GRAY2BGR)

    # oneDNN/MKLDNN dinonaktifkan karena dapat memicu error PIR pada runtime CPU Colab.
    ocr = PaddleOCR(use_textline_orientation=False, lang='en', enable_mkldnn=False)
    result = ocr.predict(prepared_image, use_doc_unwarping=False, use_textline_orientation=False)
    ocr_lines = []
    for block in result:
        texts = block.get('rec_texts', [])
        scores = block.get('rec_scores', [])
        ocr_lines.extend(text for text, score in zip(texts, scores) if score >= 0.35)

    print('Teks hasil PaddleOCR:')
    print('\n'.join(ocr_lines))
else:
    print('Set UPLOAD_RECEIPT = True untuk mengunggah dan menjalankan OCR pada foto struk.')

### Evaluasi akurasi PaddleOCR

Evaluasi OCR memerlukan *ground truth*: teks struk yang sudah diketik benar oleh peneliti. Siapkan file `ground_truth_ocr.csv` dengan kolom `filename` dan `ground_truth`, kemudian unggah CSV tersebut bersama seluruh gambar struk. Contoh:

```csv
filename,ground_truth
struk_01.jpg,"INDOMARET POP MIE AYAM 75G TOTAL 25000"
struk_02.jpg,"ALFAMART AQUA 600 ML TOTAL 17500"
```

Hasil evaluasi menampilkan CER (*Character Error Rate*), WER (*Word Error Rate*), dan *Character Accuracy*. CER/WER yang lebih kecil menunjukkan OCR lebih baik.


In [ ]:
# Evaluasi PaddleOCR terhadap ground truth manual.
# CSV harus memiliki kolom: filename, ground_truth

import os
import shutil
from pathlib import Path

EVALUATION_FOLDER = Path('ocr_evaluation_images')
EVALUATION_CSV = 'ground_truth_ocr.csv'

# Bersihkan folder unggahan evaluasi lama agar hasil tidak tercampur.
if EVALUATION_FOLDER.exists():
    shutil.rmtree(EVALUATION_FOLDER)
EVALUATION_FOLDER.mkdir()

print('Unggah ground_truth_ocr.csv dan semua gambar struk yang namanya ada di CSV.')
evaluation_uploads = files.upload()

for filename, content in evaluation_uploads.items():
    if filename.lower() == EVALUATION_CSV:
        with open(EVALUATION_CSV, 'wb') as file:
            file.write(content)
    elif filename.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
        with open(EVALUATION_FOLDER / filename, 'wb') as file:
            file.write(content)

if not Path(EVALUATION_CSV).exists():
    raise FileNotFoundError(
        'ground_truth_ocr.csv belum diunggah. File harus memiliki kolom filename dan ground_truth.'
    )

uploaded_image_paths = sorted(
    path for path in EVALUATION_FOLDER.iterdir()
    if path.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp')
)

print('File gambar yang terbaca:', [path.name for path in uploaded_image_paths])

if not uploaded_image_paths:
    raise FileNotFoundError(
        'Tidak ada gambar struk yang diunggah. Unggah CSV dan gambar dalam satu kali proses upload.'
    )

ground_truth_df = pd.read_csv(EVALUATION_CSV)

required_columns = {'filename', 'ground_truth'}
missing_columns = required_columns - set(ground_truth_df.columns)
if missing_columns:
    raise ValueError(
        f'Kolom CSV kurang: {sorted(missing_columns)}. Gunakan kolom filename dan ground_truth.'
    )

ground_truth_df = ground_truth_df.dropna(subset=['filename', 'ground_truth']).copy()
ground_truth_df['filename'] = ground_truth_df['filename'].astype(str)
ground_truth_df['ground_truth'] = ground_truth_df['ground_truth'].astype(str)

evaluation_ocr = PaddleOCR(
    use_textline_orientation=False,
    lang='en',
    enable_mkldnn=False,
)

def normalize_for_evaluation(text):
    # Kapitalisasi dan spasi tidak dihitung sebagai perbedaan OCR.
    return ' '.join(str(text).lower().split())

def levenshtein_distance(reference, hypothesis):
    # Implementasi mandiri agar notebook tidak membutuhkan library tambahan.
    if len(reference) < len(hypothesis):
        reference, hypothesis = hypothesis, reference

    previous_row = list(range(len(hypothesis) + 1))
    for i, reference_char in enumerate(reference, start=1):
        current_row = [i]
        for j, hypothesis_char in enumerate(hypothesis, start=1):
            insertions = previous_row[j] + 1
            deletions = current_row[j - 1] + 1
            substitutions = previous_row[j - 1] + (reference_char != hypothesis_char)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row

    return previous_row[-1]

def evaluate_ocr_image(image_path):
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f'Gambar tidak dapat dibaca: {image_path}')

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    enhanced = cv2.convertScaleAbs(gray, alpha=1.25, beta=5)
    prepared = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)

    result = evaluation_ocr.predict(
        prepared,
        use_doc_unwarping=False,
        use_textline_orientation=False,
    )

    lines = []
    for block in result:
        texts = block.get('rec_texts', [])
        scores = block.get('rec_scores', [])
        lines.extend(
            str(text).strip()
            for text, score in zip(texts, scores)
            if str(text).strip() and float(score) >= 0.35
        )

    return ' '.join(lines)

evaluation_results = []

for _, row in ground_truth_df.iterrows():
    filename = row['filename']
    image_path = EVALUATION_FOLDER / filename

    # Cocokkan tanpa membedakan kapitalisasi. Jika hanya satu gambar dan satu
    # baris CSV, gunakan gambar tersebut walau nama filenya berbeda.
    if not image_path.exists():
        image_path = next(
            (path for path in uploaded_image_paths if path.name.lower() == filename.lower()),
            None,
        )

    if image_path is None and len(uploaded_image_paths) == 1 and len(ground_truth_df) == 1:
        image_path = uploaded_image_paths[0]
        print(
            f'Nama CSV "{filename}" berbeda; memakai satu gambar yang tersedia: "{image_path.name}".'
        )

    if image_path is None or not image_path.exists():
        print(
            f'Lewati "{filename}": gambar tidak ditemukan. '
            f'File tersedia: {[path.name for path in uploaded_image_paths]}'
        )
        continue

    ground_truth = row['ground_truth']
    prediction = evaluate_ocr_image(image_path)

    reference = normalize_for_evaluation(ground_truth)
    hypothesis = normalize_for_evaluation(prediction)

    char_errors = levenshtein_distance(reference, hypothesis)
    word_errors = levenshtein_distance(reference.split(), hypothesis.split())

    cer = char_errors / max(len(reference), 1)
    wer = word_errors / max(len(reference.split()), 1)

    evaluation_results.append({
        'Filename': filename,
        'Ground Truth': ground_truth,
        'Hasil PaddleOCR': prediction,
        'CER': cer,
        'WER': wer,
        'Character Accuracy (%)': (1 - cer) * 100,
    })

ocr_evaluation_df = pd.DataFrame(evaluation_results)

if ocr_evaluation_df.empty:
    raise ValueError('Tidak ada gambar yang berhasil dievaluasi. Periksa nama file pada CSV.')

print('=== HASIL EVALUASI PADDLEOCR ===')
print(f'Jumlah gambar                : {len(ocr_evaluation_df)}')
print(f'Rata-rata CER                : {ocr_evaluation_df["CER"].mean():.4f}')
print(f'Rata-rata WER                : {ocr_evaluation_df["WER"].mean():.4f}')
print(f'Rata-rata Character Accuracy : {ocr_evaluation_df["Character Accuracy (%)"].mean():.2f}%')

display(ocr_evaluation_df)

ocr_evaluation_df.to_csv(
    'hasil_evaluasi_ocr.csv',
    index=False,
    encoding='utf-8-sig',
)
files.download('hasil_evaluasi_ocr.csv')


---
# Tahap 3 ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Data Preparation
Foto struk dipersiapkan dengan OpenCV, dibaca PaddleOCR, lalu teksnya disusun Gemini. Nama item hasil Gemini dinormalisasi sebelum digunakan BiLSTM.


In [ ]:
from sklearn.model_selection import train_test_split
import re

NORMALIZATION_MAP = {'cabe': 'cabai', 'mie': 'mi', 'indomiee': 'indomie', 'baksoo': 'bakso', 'capcay': 'cap cay', 'capjay': 'cap jay'}

def preprocess_text(text):
    text = re.sub(r'[^a-z\s]', ' ', str(text).lower().strip())
    text = re.sub(r'\s+', ' ', text).strip()
    return ' '.join(NORMALIZATION_MAP.get(token, token) for token in text.split())

prepared = df.dropna(subset=['text', 'category']).copy()
prepared['text_clean'] = prepared['text'].astype(str).map(preprocess_text)
prepared = prepared[prepared['text_clean'].str.len() > 0]

display(prepared[['text', 'text_clean', 'category']].head(12))
x_train, x_test, y_train, y_test = train_test_split(
    prepared['text_clean'].tolist(),
    prepared['category'].tolist(),
    test_size=CONFIG['test_size'],
    random_state=CONFIG['random_state'],
    stratify=prepared['category'].tolist(),
)
print(f'Data latih: {len(x_train)}')
print(f'Data uji  : {len(x_test)}')

---
# Tahap 4 ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Modeling
Komponen penelitian adalah PaddleOCR untuk ekstraksi teks, Gemini untuk menyusun hasil OCR menjadi transaksi terstruktur, dan Embedding ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ Bidirectional LSTM ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ Dropout ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ Softmax untuk kategori produk. Tidak ada YOLO.


### Konfigurasi Gemini

Simpan `GEMINI_API_KEY` pada Colab Secrets. Gemini menerima teks OCR dan menghasilkan JSON transaksi; BiLSTM tetap menentukan kategori.


In [ ]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError('Tambahkan GEMINI_API_KEY pada Colab Secrets.')
gemini_client = genai.Client(api_key=GEMINI_API_KEY)


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder

tf.keras.utils.set_random_seed(CONFIG['random_state'])
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=CONFIG['max_words'], oov_token='<OOV>', lower=True)
tokenizer.fit_on_texts(x_train)

def to_sequences(texts):
    sequences = tokenizer.texts_to_sequences(texts)
    return tf.keras.preprocessing.sequence.pad_sequences(sequences, maxlen=CONFIG['max_length'], padding='post', truncating='post')

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
x_train_encoded = to_sequences(x_train)

# Arsitektur klasifikasi: Embedding -> LSTM biasa -> Dropout -> Softmax.
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(CONFIG['max_length'],)),
    tf.keras.layers.Embedding(min(CONFIG['max_words'], len(tokenizer.word_index) + 1), CONFIG['embedding_dim'], mask_zero=True),
    tf.keras.layers.LSTM(CONFIG['lstm_units']),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(label_encoder.classes_), activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train_encoded, y_train_encoded, epochs=CONFIG['epochs'], batch_size=CONFIG['batch_size'], verbose=1)
model.summary()

---
# Tahap 5 ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Evaluation
BiLSTM dievaluasi dengan accuracy, precision, recall, macro F1-score, dan confusion matrix. Alur OCR ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ Gemini dievaluasi menggunakan foto struk uji terpisah dengan membandingkan kelengkapan item, qty, harga, dan total terhadap struk asli.


In [ ]:
print('Evaluasi end-to-end: bandingkan output PaddleOCR ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ Gemini dengan foto struk uji.')


In [ ]:
from sklearn.metrics import (accuracy_score, classification_report,
                             ConfusionMatrixDisplay, confusion_matrix,
                             f1_score, precision_score, recall_score)
import time

start = time.perf_counter()
probabilities = model.predict(to_sequences(x_test), verbose=0)
y_pred = label_encoder.inverse_transform(np.argmax(probabilities, axis=1))
elapsed_ms = (time.perf_counter() - start) * 1000

metrics = pd.DataFrame([{
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision macro': precision_score(y_test, y_pred, average='macro', zero_division=0),
    'Recall macro': recall_score(y_test, y_pred, average='macro', zero_division=0),
    'F1-score macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
    'Rata-rata prediksi (ms/item)': elapsed_ms / len(x_test),
}]).T.rename(columns={0: 'Nilai'})
display(metrics)

labels = list(label_encoder.classes_)
print(classification_report(y_test, y_pred, labels=labels, zero_division=0))
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred, labels=labels), display_labels=labels).plot(ax=ax, cmap='Blues', xticks_rotation=25)
ax.set_title('Confusion Matrix LSTM')
plt.tight_layout()
plt.show()

### Hasil evaluasi proyek saat ini
Nilai pada sel berikut adalah hasil evaluasi terakhir proyek lokal dan ditampilkan sebagai pembanding. Hasil eksperimen Colab adalah metrik yang muncul pada sel *Evaluation* di atas.

In [ ]:
import json

# Nilai ini berasal dari evaluasi terakhir pada proyek lokal (dataset 758 data, split 80:20).
project_metrics = {
    'Accuracy': '80.92%',
    'Precision macro': '84.31%',
    'Recall macro': '80.89%',
    'F1-score macro': '81.55%',
    'F1-score weighted': '81.53%',
}
display(pd.Series(project_metrics, name='Hasil evaluasi proyek'))

---
# Tahap 6 ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Deployment dan Uji End-to-End
Foto struk dibaca PaddleOCR, Gemini menyusun hasilnya menjadi JSON transaksi, lalu BiLSTM menentukan kategori item. Tidak ada Roboflow atau YOLO.


In [ ]:
from google.colab import files

uploaded_receipt = files.upload()
receipt_name = next(iter(uploaded_receipt))
receipt_image = cv2.imdecode(np.frombuffer(uploaded_receipt[receipt_name], np.uint8), cv2.IMREAD_COLOR)
if receipt_image is None:
    raise ValueError('Gambar struk tidak dapat dibaca.')
plt.imshow(cv2.cvtColor(receipt_image, cv2.COLOR_BGR2RGB))
plt.title('Foto struk untuk PaddleOCR')
plt.axis('off')
plt.show()


In [ ]:
# ==========================================================
# 1. OCR: FOTO STRUK -> BARIS TEKS DENGAN PADDLEOCR
# ==========================================================
# OCR seluruh foto struk dengan PaddleOCR, tanpa crop YOLO/Roboflow.
import json
receipt_ocr = PaddleOCR(use_textline_orientation=False, lang='en', enable_mkldnn=False)

def _box_position(box):
    values = np.asarray(box, dtype=float).flatten()
    if values.size == 4:
        x1, y1, x2, y2 = values
        return x1, (y1 + y2) / 2, max(1, abs(y2 - y1))
    points = values.reshape(-1, 2)
    return points[:, 0].min(), points[:, 1].mean(), max(1, np.ptp(points[:, 1]))

def ocr_to_rows(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    enhanced = cv2.convertScaleAbs(gray, alpha=1.25, beta=5)
    prepared = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
    result = receipt_ocr.predict(prepared, use_doc_unwarping=False, use_textline_orientation=False)
    entries = []
    for block in result:
        texts = block.get('rec_texts', [])
        scores = block.get('rec_scores', [])
        boxes = block.get('rec_boxes', [])
        for index, (text, score) in enumerate(zip(texts, scores)):
            if score < 0.35 or not str(text).strip():
                continue
            x, y, h = _box_position(boxes[index]) if index < len(boxes) else (index * 100, index * 30, 20)
            entries.append({'text': str(text).strip(), 'x': x, 'y': y, 'h': h})
    if not entries:
        return []
    entries.sort(key=lambda entry: entry['y'])
    tolerance = max(12, np.median([entry['h'] for entry in entries]) * 0.75)
    rows = []
    for entry in entries:
        row = next((r for r in rows if abs(entry['y'] - r['y']) <= tolerance), None)
        if row is None:
            rows.append({'y': entry['y'], 'entries': [entry]})
        else:
            row['entries'].append(entry)
            row['y'] = np.mean([item['y'] for item in row['entries']])
    return [' '.join(item['text'] for item in sorted(row['entries'], key=lambda item: item['x'])) for row in rows]

def to_rupiah(value):
    digits = re.sub(r'[^0-9]', '', str(value))
    return int(digits) if digits else None



item_rows = ocr_to_rows(receipt_image)
total_rows = []
info_rows = []
ocr_text = '\n'.join(item_rows)



merchant = ''
supplier = ''
# ==========================================================
# 2. GEMINI: BARIS OCR -> JSON ITEM STRUK TERSTRUKTUR
# ==========================================================
gemini_prompt = f"""
Anda adalah parser struk belanja Indonesia. Ubah teks OCR struk berikut menjadi JSON valid saja, tanpa markdown dan tanpa penjelasan tambahan.

Format JSON wajib:
{{
  "merchant": "",
  "transaction_date": null,
  "items": [{{"name": "", "qty": 1, "price": null, "amount": null}}],
  "total": null
}}

Aturan penting:
- Ambil hanya produk atau barang yang benar-benar dibeli.
- Perbaiki kesalahan OCR pada nama produk apabila koreksinya jelas dari konteks produk Indonesia, merek, ukuran, berat, harga, atau baris sekitar.
- Keluarkan nama produk yang rapi, normal, dan mudah dibaca dalam Bahasa Indonesia; jangan mempertahankan typo OCR yang jelas.
- Contoh koreksi: "POP MIE AYAM 75G" -> "Pop Mie Ayam 75g"; "POP MIE PD.DWR AYM75" -> "Pop Mie Pedas Dower Ayam 75g"; "JLTRA KCNG HIJAU 250" -> "Ultra Kacang Hijau 250ml"; "NUTRIJEL PWD.STRW.15" -> "Nutrijell Powder Strawberry 15g"; "Knzler Sngles Kju 65" -> "Kanzler Singles Keju 65g"; "ANZLR BAKSO ORI 48G" -> "Kanzler Bakso Original 48g".
- Jangan memasukkan subtotal, pajak, diskon, pembayaran, kembalian, nomor transaksi, atau nama toko sebagai item.
- Jika qty tidak terbaca, isi 1. Jika harga atau jumlah tidak terbaca, isi null.
- Jangan menentukan kategori barang dan jangan membuat item baru yang tidak ada pada struk.
- Jika koreksi nama benar-benar tidak yakin, pertahankan nama OCR yang paling masuk akal, tetapi rapikan kapitalisasi dan spasinya.

Teks OCR:
{ocr_text}
"""
response = gemini_client.models.generate_content(model='gemini-3.6-flash', contents=gemini_prompt, config={'response_mime_type': 'application/json'})
structured_receipt = json.loads(response.text)
print('\n=== JSON HASIL GEMINI ===')
print(json.dumps(structured_receipt, indent=2, ensure_ascii=False))
merchant = structured_receipt.get('merchant') or 'Tidak diketahui'
transaction_date = structured_receipt.get('transaction_date')
parsed_items = structured_receipt.get('items', [])

# ==========================================================
# 3. LSTM: NAMA ITEM DARI GEMINI -> KATEGORI PRODUK
# model adalah LSTM biasa yang sudah dilatih pada Tahap 4.
# ==========================================================
final_items = []
for item in parsed_items:
    item['name'] = str(item.get('name', '')).strip()
    if not item['name']:
        continue
    clean_name = preprocess_text(item['name'])
    # Prediksi LSTM menghasilkan probabilitas untuk setiap kategori.
    probabilities = model.predict(to_sequences([clean_name]), verbose=0)[0]
    confidence = float(np.max(probabilities))
    category = label_encoder.inverse_transform([int(np.argmax(probabilities))])[0]
    final_items.append({
        'Nama produk': item['name'],
        'Kategori': category if confidence >= CONFIG['minimum_confidence'] else 'lainnya',
        'Confidence (%)': round(confidence * 100, 2),
        'Qty': item.get('qty') or 1,
        'Harga satuan': to_rupiah(item.get('price')),
        'Jumlah': to_rupiah(item.get('amount')),
    })

items_df = pd.DataFrame(final_items)
receipt_total = to_rupiah(structured_receipt.get('total'))
items_total = int(pd.to_numeric(items_df['Jumlah'], errors='coerce').fillna(0).sum()) if not items_df.empty else 0

print('=== INFORMASI MERCHANT / SUPPLIER ===')
print(f'Merchant : {merchant or "Tidak terdeteksi"}')
print(f'Supplier : {supplier or "Tidak terdeteksi"}')
print('Header OCR:')
print(' | '.join(info_rows) or 'Tidak ada area info_toko')

print('=== Baris hasil OCR pada area item ===')
print('\n'.join(item_rows))
print('\n=== Item terstruktur untuk aplikasi ===')
display(items_df)
print(f'Jumlah item: Rp{items_total:,}')
if receipt_total is not None:
    difference = items_total - receipt_total
    print(f'Total pada struk: Rp{receipt_total:,}')
    print(f'Selisih item vs total: Rp{difference:,}')
    if difference != 0:
        print('PERINGATAN: Ada item yang belum ter-parse, diskon, atau biaya tambahan. Periksa tabel sebelum disimpan.')

# Struktur siap dikirim ke endpoint/backend atau disimpan ke database.
receipt_payload = {
    'merchant': merchant,
    'supplier': supplier, 
    'raw_ocr_text': ocr_text,
    'items': items_df.to_dict(orient='records'),
    'items_total': items_total,
    'total': receipt_total if receipt_total is not None else items_total,
}
receipt_payload


## Integrasi ke aplikasi
Flutter mengirim foto ke endpoint `POST /scan`. Backend menjalankan OpenCV ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ PaddleOCR ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ Gemini untuk struktur transaksi ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ BiLSTM untuk kategori item. Hasil yang belum lengkap tetap dapat dikoreksi pengguna sebelum disimpan.


In [ ]:
examples = ['Mie Sedaap Goreng', 'Aqua 600ML', 'Beras premium 5 kg', 'Sabun cuci']
for item in examples:
    probability = model.predict(to_sequences([preprocess_text(item)]), verbose=0)[0]
    confidence = float(np.max(probability))
    category = label_encoder.inverse_transform([int(np.argmax(probability))])[0]
    category = category if confidence >= CONFIG['minimum_confidence'] else 'lainnya'
    print(f'{item:24} -> {category:28} ({confidence:.2%})')

## Kesimpulan dan batasan
Model BiLSTM dapat membantu otomatisasi pengelompokan nama produk dari struk pada aplikasi Keuangan UMKM. Kinerja dipengaruhi oleh jumlah, keseimbangan, dan variasi data latih. Untuk pengembangan selanjutnya, tambahkan data produk baru, lakukan evaluasi berkala, dan pertimbangkan koreksi pengguna sebagai data pelatihan tambahan.